<h1>HEDGE FUND COMPETITION</h1>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<h2>1. EXPLORATORY DATA ANALYSIS</h2>

In [ ]:
import pandas as pd

train = pd.read_parquet("/kaggle/input/ts-forecasting/train.parquet")
test  = pd.read_parquet("/kaggle/input/ts-forecasting/test.parquet")

print(train.shape)
print(test.shape)

In [ ]:
# Basic inspection
print("\nTrain dtypes:")
print(train.dtypes)

print("\nTest dtypes:")
print(test.dtypes)

**OUTPUT VARIABLES** <br>
* y_target (ts_index,horizon): The future value given what knew at time “ts_index”, the forecast in given “horizon” is y_target <br>
Example: 
ts_index = 89
horizon = 25
y_target = -0.5513 <br>
“Given what we knew at time 89, the long-term (horizon = 25) forecast outcome turned out to be −0.5513.” <br>
* y-target is a continuous value = > This might be a regression problem

In [ ]:
train.head()

**PREDICTORS, RESPONSE VARIABLES, IDENTIFY**<br>
X = features at time ts_index (86 features in total)<br>
y = y_target (future outcome for a given horizon)<br>
Define/Identities of each observation: code, sub_code, sub_category

In [ ]:
# Missing values
print("\nMissing values in train:")
print(train.isna().sum().sort_values(ascending=False).head(20))

print("\nMissing values in test:")
print(test.isna().sum().sort_values(ascending=False).head(20))

**MISSING VALUE ASSESSMENT** <br>
* For Train set missing value: The worst missing data is feature_at with 665676, which is 12,5% of the data set. This missing level is just moderate
* Test set missing value: The worst missing data is feature_at with 558243, which is 38,5% of the data set. This number is high, but we are not gonna build model based on the test set.
* The missing pattern between the train set and the test set is not the same => Suggested model to handle missing data: Light Gradient Boosting Machine or Linear Regression

In [ ]:
# Target distribution
print("\ny_target summary:")
print(train["y_target"].describe())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(train["y_target"], bins=100)
plt.title("y_target distribution")
plt.show()

**y-targer DISTRIBUTION** <br>
* Centered near zero (75% of the data is between -0.13 and 0.05) <br>
* Very heavy tails (min=-2201, max=2314) => RMSE can be dominated by outliers, model instability possible, linear models will struggle => Light GBM is much better <br>
(There is a massive gap between the majority of the data around the center and the tails.)

In [ ]:
# Horizon distribution
print("\nHorizon counts:")
print(train["horizon"].value_counts())

**HORIZONTAL DISTRIBUTION** <br>
* The distribution is balanced

In [ ]:
# Unique entities
print("\nUnique codes:", train["code"].nunique())
print("Unique sub_codes:", train["sub_code"].nunique())
print("Unique sub_categories:", train["sub_category"].nunique())

**Enities**
* Small number of main groups, but many sub_code => at each time step, multiple entities exist
* Strong cross-sectional structure (many entities are compared at the same time) => Relationship between entities matters

In [ ]:
print(train.groupby("code")["ts_index"].agg(["min", "max", "nunique"]).head())

**TIME STRUCTURE**
* min ts_index = 1
* max ts_index = 3601
* number of unique per code ~ 2900 - 3400 => Each code span ~3000+ time steps => Long history => Good

In [ ]:
train.sort_values(["code", "ts_index", "horizon"]).head(20)

**OVERALL CHECK** <br>
We can see that for the same (code, ts_index) -> multiple horizons exist (1,2,10,25,...)

**BIG PICTURE SUMMARY** <br>
* It could be Regression Problems <br>
* Heavy-tailed target <br>
* Balanced Horizons <br>
* Moderate missing <br>
* Multi-horizons per time step <br>
* Crossectional dominant <br>

<h2>2. VALIDATION STRATEGY</h2>

In [ ]:
print(train["ts_index"].max())
print(test["ts_index"].max())

In [ ]:
print(train["ts_index"].min())
print(test["ts_index"].min())

This is a pure forward time_split. The test set is the future.

**DATA SPLIT**

In [ ]:
cutoff = 3400

train_data = train[train["ts_index"] <= cutoff].copy()
valid_data = train[train["ts_index"] > cutoff].copy()

print(train_data.shape)
print(valid_data.shape)

<h2>3. MODELING SETUP</h2>

In [ ]:
# Define features and target
TARGET = "y_target"
DROP_COLS = ["id", "y_target", "weight"]

FEATURES = [col for col in train.columns if col not in DROP_COLS]

print("Number of features:", len(FEATURES))

# Prepare Data
X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET]

X_valid = valid_data[FEATURES].copy()
y_valid = valid_data[TARGET]

w_valid = valid_data["weight"]  # For weighted evaluation

#Convert Categorical data
cat_cols = ["code", "sub_code", "sub_category", "horizon"]

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

In [ ]:
# Install lightgbm
!pip install lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

# Train baseline model
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols
)

# Evaluate the model
preds = model.predict(X_valid)

weighted_mse = np.average((preds - y_valid) ** 2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

**RESULT INTERPRETATION**
* We're training on around 5M rows with 91 features (no features being dropped)
* The starting point is approximately the mean (before earning anything, the model predicts y_target = -0.666 for every row)
* Weighted RMSE is approximately 0.12, which is really small => this shows that extreme values(outliers) are rare and have small weight

**MODEL DIAGNOSTIC**

In [ ]:
#Horizontal Breakdown
valid_data["preds"] = preds

for h in sorted(valid_data["horizon"].unique()):
    subset = valid_data[valid_data["horizon"] == h]
    rmse = np.sqrt(np.average(
        (subset["preds"] - subset["y_target"])**2,
        weights=subset["weight"]
    ))
    print(f"Horizon {h} RMSE:", rmse)

**Interpretation:**
* Errors increase with Horizon => Good
* Features contain strong short-term signal => long-term might be weaker
* Horizon 1 & 3 RMSE is smaller than the overall weighted RMSE (~0.12)

<h2>4. TUNING MODEL</h2>

In [ ]:
# Update model with n_estimators=5000 (larger numbers of trees)
from lightgbm import early_stopping, log_evaluation

model = lgb.LGBMRegressor(
    n_estimators=5000,     # large number
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(200),   # stop if no improvement for 200 rounds
        log_evaluation(200)
    ]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

* The new weighted RMSE (n_estimators: 5000) is larger than the previous weighted RMSE (n-estimators: 500)

In [ ]:
# Update model with num_leaves = 128 (larger numbers of leaves)
from lightgbm import early_stopping, log_evaluation

model = lgb.LGBMRegressor(
    n_estimators=500,     # large number
    learning_rate=0.05,
    num_leaves=128,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(200),   # stop if no improvement for 200 rounds
        log_evaluation(200)
    ]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

* The new weighted RMSE (num_leaves = 128) is smaller than the previous weighted RMSE (num-leaves = 64) => Good sign, massive improvement

In [ ]:
#Should we push further the num_leaves to 256?
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation

model = lgb.LGBMRegressor(
    n_estimators=500,     
    learning_rate=0.05,
    num_leaves=256, # larger leaves
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(200),   # stop if no improvement for 200 rounds
        log_evaluation(200)
    ]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

**Interpret the result**
We can see a massive improvement in weighted RMSE. Best iteration is 270 => Larger tree learn faster. Early stopping caught it correctly, the model may start overfitting after 270
Note: We still have to be careful by checking 
    1.	Is weighted evaluation correct?
	2.	Are weights passed consistently?
	3.	Is the validation split still correct?
	4.	Any leakage accidentally introduced?

**Check if we can go further with num_leaves of 256**

In [ ]:
#Step 1: Run Horizontal Breakdown to check
valid_eval = valid_data.copy()
valid_eval["preds"] = preds

print("Horizon-wise Weighted RMSE:")

for h in sorted(valid_eval["horizon"].unique()):
    subset = valid_eval[valid_eval["horizon"] == h]
    
    rmse = np.sqrt(
        np.average(
            (subset["preds"] - subset["y_target"])**2,
            weights=subset["weight"]
        )
    )
    
    print(f"Horizon {h}: {rmse:.6f}")

We can notice that H25 is now lower than H10 => non-linear pattern => a strong sign your model is learning deeper interactions

In [ ]:
# ===== Step 2: Unweighted RMSE =====

rmse_unweighted = np.sqrt(np.mean((preds - y_valid)**2))
print("Unweighted RMSE:", rmse_unweighted)

Huge compared to weighted RMSE (~0.079) 

In [ ]:
# ===== Step 3: num_leaves = 512 =====

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation

model_512 = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=512,
    random_state=42
)

model_512.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[
        early_stopping(200),
        log_evaluation(200)
    ]
)

print("Best iteration (512 leaves):", model_512.best_iteration_)

# Evaluate weighted RMSE
preds_512 = model_512.predict(X_valid)

weighted_rmse_512 = np.sqrt(
    np.average(
        (preds_512 - y_valid)**2,
        weights=w_valid
    )
)

print("Weighted RMSE (512 leaves):", weighted_rmse_512)

**Interpretation:**
	•	256 captures useful nonlinear structure <br>
	•	512 starts overfitting <br>
	•	Validation loss increases <br>
	•	Early stopping kicks in earlier <br>
=> 256 leaves is currently optimal. <br>
Go further with this model:
"n_estimators=500,     
learning_rate=0.05,
num_leaves=256, 
random_state=42"

<h2>REGULARIZATION TUNING</h2>
We will try to control overfitting

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=256,
    random_state=42,
    min_child_samples=100,
    feature_fraction=0.8
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[early_stopping(200)]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

In [ ]:
print("Train target mean:", y_train.mean())
print("Valid target mean:", y_valid.mean())
print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Best iteration:", model.best_iteration_)

In [ ]:
train["y_target"].describe()
train_data["y_target"].describe()

**Interpretation:** <br>
Heavy regularization is not helpful here. The current best model: <br>
num_leaves = 256 <br>
learning_rate = 0.05 <br>
n_estimators = 500 <br>
no min_child_samples tuning <br>
no feature_fraction <br>
no sample_weight <br>

**Next step:**
* Try increase Trees + Reduce Learning Rate
* Try Slightly Higher Leaves (Carefully)

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=256,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[early_stopping(200)]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

The performance get worse compare to the current best model => stop playing with learning rate => test controlled increase in capacity, not reduction.<br>
Next try: <br>  
* n_estimators=1000
* learning_rate=0.05
* num_leaves=300
* random_state=42

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=300,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[early_stopping(200)]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

The weighted RMSE is not better than the current best mode. <br>
=> The best model still: <br>
num_leaves = 256 <br>
learning_rate = 0.05 <br>
RMSE ≈ 0.07905 

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=1200,
    learning_rate=0.04,
    num_leaves=256,
    min_child_samples=30,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    categorical_feature=cat_cols,
    callbacks=[early_stopping(200)]
)
print("Best iteration:", model.best_iteration_)

#Recompute Weighted RMSE
preds = model.predict(X_valid)
weighted_mse = np.average((preds - y_valid)**2, weights=w_valid)
weighted_rmse = np.sqrt(weighted_mse)

print("Weighted RMSE:", weighted_rmse)

Still cannot compare to the current best model 

<h2>5. MODEL RUNNING</h2>

In [ ]:
Implement on another notebook due to system memory